## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample
from statistics import mean
import warnings
import math

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from scipy.signal import get_window
from scipy.fft import rfft, rfftfreq
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = [
    "TB-zero_threshold",
    # "TB-twentyfive_threshold",
    "TB-twentyfive_again_threshold",
    "TB-fifty_threshold",
    "TB-seventyfive_threshold",
    "TB-hundred_threshold",
    "TB-hundred_twentyfive_threshold",
    "TB-hundred_fifty_threshold",
    "TB-hundred_sevetyfive_threshold",  # spelling is correct
    "TB-twohundred_threshold",
    "TB-twohundred_twentyfive_threshold",
    "TB-twohundred_fifty_threshold",
    "TB-twohundred_seventyfive_threshold",
    "TB-threehundred_threshold",
    "TB-fourhundred_threshold",
]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Histogram Creation

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_class_col_name).copy()

    print(psd_report.shape)
    print(neutrons_only.shape)

    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"].astype("int32")
    # neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
    
    signals_np = signals_df.to_numpy()
    baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
    psd_report["peak_height"] = signals_df.max(axis=1)
    exp_data["signals_df"] = signals_df

In [ ]:
height_bin_width = 100
energy_bin_width = 50
for exp_id, exp_data in experiment_neutron_data.items():
    # print(exp_id)
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    # print(neutrons_only.head())
    energy = psd_report["ENERGY"]
    peak_height = psd_report["peak_height"]

    # print(energy.min(), energy.max())
    # print(peak_height.min(), peak_height.max())

    height_bins = np.arange(0, 15500, step=height_bin_width)
    energy_bins = np.arange(0, 3500, step=energy_bin_width)
    
    Zh, *_ = np.histogram(peak_height, bins=height_bins)
    Ze, *_ = np.histogram(energy, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "all": {
            "height": Zh,
            "height_bins": height_bins,
            "energy": Ze,
            "energy_bins": energy_bins
        },
    }

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    for key in ["height", "energy"]:
        Z = phd_data[key]
        Z_total = np.sum(Z)
        Z_fraction = Z / Z_total
        phd_data[f"{key}_fraction"] = Z_fraction

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.subplots_adjust(wspace=0.2)
dataset_whitelist = [
    # "TB-zero_threshold",
    # "TB-twentyfive_threshold",
    "TB-twentyfive_again_threshold",
    # "TB-fifty_threshold",
    # "TB-seventyfive_threshold",
    # "TB-hundred_threshold",
    # "TB-hundred_twentyfive_threshold",
    # "TB-hundred_fifty_threshold",
    # "TB-hundred_sevetyfive_threshold",  # spelling is correct
    # "TB-twohundred_threshold",
    # "TB-twohundred_twentyfive_threshold",
    "TB-twohundred_fifty_threshold",
    # "TB-twohundred_seventyfive_threshold",
    # "TB-threehundred_threshold",
    # "TB-fourhundred_threshold",
]
zorders = {
    # "TB-zero_threshold": 0,
    # "TB-twentyfive_threshold",
    "TB-twentyfive_again_threshold": 0,
    # "TB-fifty_threshold": 0,
    # "TB-seventyfive_threshold",
    # "TB-hundred_threshold": 0,
    # "TB-hundred_twentyfive_threshold",
    # "TB-hundred_fifty_threshold": 0,
    # "TB-hundred_sevetyfive_threshold",  # spelling is correct
    # "TB-twohundred_threshold": 0,
    # "TB-twohundred_twentyfive_threshold",
    "TB-twohundred_fifty_threshold": 1,
    # "TB-twohundred_seventyfive_threshold",
    # "TB-threehundred_threshold": 0,
    # "TB-fourhundred_threshold": 0,
}

# color = bg_blue
# alpha = 0.1
# alpha = 1
for exp_id, exp_data in experiment_neutron_data.items():
    if exp_id not in dataset_whitelist:
        continue
    color = bg_blue if exp_id == "TB-twohundred_fifty_threshold" else bg_red
    alpha = 1 if exp_id == "TB-twohundred_fifty_threshold" else 0.5
    zorder_mod = zorders[exp_id]
    
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    bins = phd_histogram_data["height_bins"]
    histo_counts = phd_histogram_data["height"]

    energy_bin_mids = (bins[1:] + bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(
        energy_bin_mids,
        histo_counts,
        label=exp_id,
        alpha=alpha,
        zorder=5+zorder_mod,
        color=color
    )
ax.set_xlim(0, 15500)
ax.set_ylim(0, 5000)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
# ax.set_yscale("log")
# ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.subplots_adjust(wspace=0.2)

for exp_id, exp_data in experiment_neutron_data.items():
    color = bg_blue
    alpha = 0.1
    
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    bins = phd_histogram_data["energy_bins"]
    histo_counts = phd_histogram_data["energy"]

    energy_bin_mids = (bins[1:] + bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(energy_bin_mids, histo_counts, label=exp_id, alpha=alpha)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
ax.set_yscale("log")
ax.legend()

In [ ]:
ncols = 2
nrows = math.ceil(len(experiment_neutron_data.items()) / ncols)

In [ ]:
fig, axs = plt.subplots(ncols=ncols, nrows=nrows, sharex=True, figsize=(12*ncols, 8*nrows))
# fig.subplots_adjust(wspace=0.2, hspace=0.2)
axs = axs.ravel()
for ax in axs:
    ax.axis("off")

color = bg_blue
# alpha = 0.1
for ax, (exp_id, exp_data) in zip(axs, experiment_neutron_data.items()):
    ax.axis("on")
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    bins = phd_histogram_data["height_bins"]
    histo_counts = phd_histogram_data["height"]

    energy_bin_mids = (bins[1:] + bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(energy_bin_mids, histo_counts, color=color)
    ax.set_title(exp_id)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("Counts", fontsize=fontsize)
    ax.set_xlim(0, 15500)
    ax.set_ylim(0, None)
    # ax.set_yscale("log")
    ax.grid()
# ax.legend()

In [ ]:
fig, axs = plt.subplots(ncols=ncols, nrows=nrows, sharex=True, figsize=(12*ncols, 8*nrows))
# fig.subplots_adjust(wspace=0.2, hspace=0.2)
axs = axs.ravel()
for ax in axs:
    ax.axis("off")

color = bg_blue
# alpha = 0.1
for ax, (exp_id, exp_data) in zip(axs, experiment_neutron_data.items()):
    ax.axis("on")
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    bins = phd_histogram_data["energy_bins"]
    histo_counts = phd_histogram_data["energy"]

    energy_bin_mids = (bins[1:] + bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(energy_bin_mids, histo_counts, color=color)
    ax.set_title(exp_id)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Pulse energy (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("Counts", fontsize=fontsize)
    # ax.set_yscale("log")
    ax.set_xlim(0, 3500)
    ax.set_ylim(0, None)
    ax.grid()
# ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.subplots_adjust(wspace=0.2)

dataset_whitelist = [
    "TB-zero_threshold",
    # "TB-twentyfive_threshold",
    # "TB-twentyfive_again_threshold",
    "TB-fifty_threshold",
    # "TB-seventyfive_threshold",
    "TB-hundred_threshold",
    # "TB-hundred_twentyfive_threshold",
    "TB-hundred_fifty_threshold",
    # "TB-hundred_sevetyfive_threshold",  # spelling is correct
    "TB-twohundred_threshold",
    # "TB-twohundred_twentyfive_threshold",
    "TB-twohundred_fifty_threshold",
    # "TB-twohundred_seventyfive_threshold",
    "TB-threehundred_threshold",
    "TB-fourhundred_threshold",
]
color = bg_blue
alpha = 0.5
                     
for exp_id, exp_data in experiment_neutron_data.items():
    if exp_id not in dataset_whitelist:
        continue
    
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    bins = phd_histogram_data["height_bins"]
    histo_frac = phd_histogram_data["height_fraction"]

    bin_mids = (bins[1:] + bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(bin_mids, histo_frac, label=exp_id, alpha=alpha)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
ax.set_ylabel("Fractional counts", fontsize=fontsize)
# ax.set_yscale("log")
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.subplots_adjust(wspace=0.2)

dataset_whitelist = [
    "TB-zero_threshold",
    # "TB-twentyfive_threshold",
    # "TB-twentyfive_again_threshold",
    # "TB-fifty_threshold",
    # "TB-seventyfive_threshold",
    "TB-hundred_threshold",
    # "TB-hundred_twentyfive_threshold",
    # "TB-hundred_fifty_threshold",
    # "TB-hundred_sevetyfive_threshold",  # spelling is correct
    "TB-twohundred_threshold",
    # "TB-twohundred_twentyfive_threshold",
    # "TB-twohundred_fifty_threshold",
    # "TB-twohundred_seventyfive_threshold",
    "TB-threehundred_threshold",
    "TB-fourhundred_threshold",
]
color = bg_blue
alpha = 0.5
                     
for exp_id, exp_data in experiment_neutron_data.items():
    if exp_id not in dataset_whitelist:
        continue
    
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    bins = phd_histogram_data["energy_bins"]
    histo_frac = phd_histogram_data["energy_fraction"]

    bin_mids = (bins[1:] + bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(bin_mids, histo_frac, label=exp_id, alpha=alpha)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel("Pulse energy (ADC channel)", fontsize=fontsize)
ax.set_ylabel("Fractional counts", fontsize=fontsize)
# ax.set_yscale("log")
ax.legend()

In [ ]:
input("Processing done, hit Enter to finish")
stop()